In [ ]:
%pip install pandas matplotlib seaborn  # Décommente si ces bibliothèques ne sont pas installées

import json
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration esthétique
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

# 1. Chargement des résultats MapReduce
json_path = '../results/rentabilite_horaire.json'

if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Transformation en DataFrame Pandas
    # Gestion du format (liste de dictionnaires ou dictionnaire de clés)
    if isinstance(data, dict):
        df = pd.DataFrame.from_dict(data, orient='index')
        df['heure'] = df.index.astype(int)
    else:
        df = pd.DataFrame(data)
        
    df = df.sort_values('heure').reset_index(drop=True)
    
    # 2. Graphique 1 : Courbe de Rentabilité Horaire ($/km)
    plt.figure(figsize=(12, 5))
    plt.plot(df['heure'], df['gain_km'], marker='o', color='#e74c3c', linewidth=2.5, markersize=8)
    plt.axhline(df['gain_km'].mean(), color='gray', linestyle='--', label=f"Moyenne globale ({df['gain_km'].mean():.2f} $/km)")
    
    plt.title('Analyse de la Rentabilité Horaire (Gain Moyen par Kilomètre)', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Heure de prise en charge (Tranche horaire)', fontsize=12)
    plt.ylabel('Ratio : Recette / Distance ($ / km)', fontsize=12)
    plt.xticks(range(0, 24))
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()
    
    # Création du dossier images s'il n'existe pas pour le rapport
    os.makedirs('../docs/images', exist_ok=True)
    plt.savefig('../docs/images/rentabilite_horaire.png', dpi=300, bbox_inches='tight')
    plt.show()

    # 3. Graphique 2 : Volume d'activité par tranche horaire
    plt.figure(figsize=(12, 5))
    sns.barplot(x='heure', y='nb_courses', data=df, color='#3498db', alpha=0.8)
    plt.title("Volume d'activité : Nombre total de courses par heure (Distribution temporelle)", fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Heure de la journée', fontsize=12)
    plt.ylabel('Nombre de trajets traités', fontsize=12)
    plt.savefig('../docs/images/volume_activite.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Graphiques générés avec succès et sauvegardés dans docs/images/")
else:
    print(f"❌ Erreur : Le fichier {json_path} est introuvable. Exécute d'abord ton script PySpark.")